In [1]:
# ============================================================================
# Cell 1: 导入库
# ============================================================================
from vnpy.alpha.lab import AlphaLab
from vnpy.trader.constant import Interval
import polars as pl
from pathlib import Path
from datetime import datetime,timedelta
from vnpy.alpha import Segment, AlphaDataset
import pandas as pd
import numpy as np
import lightgbm as lgb
from vnpy.factor_define import (
    FACTOR_REGISTRY,
    FACTOR_NAMES,

)
import pickle
import gc

In [2]:
# ============================================================================
# Cell 2: 路径配置和AlphaLab创建
# ============================================================================
vt_index_symbol = "000300.SSE"
BASE_PATH = Path('D:/Aquant project/MF')
LAB_PATH = BASE_PATH / 'MF_lab'

# 获取MF_Lab
lab = AlphaLab(str(LAB_PATH))

In [3]:
# ============================================================================
# Cell 3: 时间配置
# ============================================================================
# 总时间跨度
start = datetime(2018, 1, 1)
end = datetime(2026,5,8)
interval1 = Interval.MINUTE                  #数据频率

# 回测跨度 回测需要日线数据算收益
test_start = datetime(2025, 1, 1)
test_end = end
interval2 = Interval.DAILY

# 训练跨度
train_start = datetime(2018, 1, 1)
train_end = datetime(2023, 12, 31)

# 验证跨度
valid_start = datetime(2024, 1, 1)
valid_end = datetime(2024, 12, 31)

# 加载成分股代码
component_symbols = lab.load_component_symbols(vt_index_symbol, start, end)

In [4]:
# ============================================================================
# Cell 4: 加载数据集
# ============================================================================
DATASET_NAME = 'v100'
dataset: AlphaDataset = lab.load_dataset(DATASET_NAME)

In [5]:
print(dataset.learn_df)

shape: (604_800, 68)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ datetime  ┆ vt_symbol ┆ late_skew ┆ down_vol_ ┆ … ┆ daily_ran ┆ effective ┆ adjusted_ ┆ label    │
│ ---       ┆ ---       ┆ _ret      ┆ perc      ┆   ┆ ge        ┆ _spread   ┆ range     ┆ ---      │
│ datetime[ ┆ str       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ f64      │
│ μs]       ┆           ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 2018-01-0 ┆ 000001.SZ ┆ 0.206012  ┆ -0.304682 ┆ … ┆ 1.109024  ┆ -0.027224 ┆ 0.666616  ┆ -1.67721 │
│ 2         ┆ SE        ┆           ┆           ┆   ┆           ┆           ┆           ┆ 3        │
│ 00:00:00  ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ 2018-01-0 ┆ 000002.SZ ┆ -0.013581 ┆ 0.642969  ┆ … ┆ 1.30848   ┆ -0.1

In [6]:
print(dataset.result_df)

shape: (1_051_119, 69)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ datetime  ┆ vt_symbol ┆ open      ┆ late_skew ┆ … ┆ daily_ran ┆ effective ┆ adjusted_ ┆ label    │
│ ---       ┆ ---       ┆ ---       ┆ _ret      ┆   ┆ ge        ┆ _spread   ┆ range     ┆ ---      │
│ datetime[ ┆ str       ┆ f64       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ f64      │
│ μs]       ┆           ┆           ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 2018-01-0 ┆ 000001.SZ ┆ 10.551859 ┆ 0.16691   ┆ … ┆ 0.045693  ┆ 0.20726   ┆ 0.019476  ┆ -0.05025 │
│ 2         ┆ SE        ┆           ┆           ┆   ┆           ┆           ┆           ┆ 5        │
│ 00:00:00  ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ 2018-01-0 ┆ 000002.SZ ┆ 23.764671 ┆ 0.062675  ┆ … ┆ 0.048967  ┆ 0.

In [7]:
print(dataset.infer_df)

shape: (604_800, 68)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ datetime  ┆ vt_symbol ┆ late_skew ┆ down_vol_ ┆ … ┆ daily_ran ┆ effective ┆ adjusted_ ┆ label    │
│ ---       ┆ ---       ┆ _ret      ┆ perc      ┆   ┆ ge        ┆ _spread   ┆ range     ┆ ---      │
│ datetime[ ┆ str       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ f64      │
│ μs]       ┆           ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 2018-01-0 ┆ 000001.SZ ┆ 0.206012  ┆ -0.299025 ┆ … ┆ 1.109024  ┆ -0.027224 ┆ 0.666616  ┆ -1.67721 │
│ 2         ┆ SE        ┆           ┆           ┆   ┆           ┆           ┆           ┆ 3        │
│ 00:00:00  ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ 2018-01-0 ┆ 000002.SZ ┆ -0.013581 ┆ 0.631031  ┆ … ┆ 1.30848   ┆ -0.1

In [8]:
# ============================================================================
# Cell 4 修改版：提取 LambdaRank 所需的数据（按天等频分档）
# ============================================================================
def extract_lambdarank_data(dataset, segment, n_quantiles=5):
    """
    从 AlphaDataset 提取 LambdaRank 所需数据（稳健版）
    使用 Polars 的 rank + 线性映射，避免 qcut 问题
    """
    df = dataset.fetch_learn(segment)
    df = df.sort('datetime')

    # 方案：每天内按收益率排序，然后等频分成 n_quantiles 档
    # 使用 rank('ordinal') 得到每个样本在当天的唯一排名（从1开始）
    df = df.with_columns(
        pl.col('label').rank('ordinal').over('datetime').alias('_rank')
    )
    # 计算每天的总样本数
    df = df.with_columns(
        pl.col('label').count().over('datetime').alias('_day_count')
    )
    # 将排名映射到 0 ~ n_quantiles-1
    # 公式：floor( (rank - 1) / (day_count - 1) * (n_quantiles - 1) )
    # 注意：当 day_count == 1 时，分母为0，需要特殊处理
    df = df.with_columns(
        pl.when(pl.col('_day_count') == 1)
        .then(n_quantiles // 2)   # 只有一只股票时给中间档位
        .otherwise(
            ((pl.col('_rank') - 1) / (pl.col('_day_count') - 1) * (n_quantiles - 1))
            .cast(pl.Int64)
        )
        .alias('rank_label')
    )

    # 验证标签是否在 0 ~ n_quantiles-1 范围内
    print(f"{segment.name}: 标签唯一值 = {df['rank_label'].unique().to_numpy()}")

    # 提取特征和元数据
    meta_cols = ['datetime', 'vt_symbol']
    df_meta = df.select(meta_cols)
    exclude_cols = ['datetime', 'vt_symbol', 'label', 'rank_label', '_rank', '_day_count']
    feature_cols = [c for c in df.columns if c not in exclude_cols]

    X = df.select(feature_cols).to_numpy()
    y = df['rank_label'].to_numpy()
    date_codes = df['datetime'].to_numpy()
    unique_dates, group_sizes = np.unique(date_codes, return_counts=True)

    print(f'{segment.name}: X.shape={X.shape}, y 取值 {np.unique(y)}')
    print(f'交易日数量 = {len(unique_dates)}, 平均每天样本数 = {group_sizes.mean():.1f}')
    return X, y, df_meta, group_sizes

# 提取数据（注意变量名保持一致）
n_quantiles = 30
print('提取 LambdaRank 训练数据...')
X_train, y_train, meta_train, group_train = extract_lambdarank_data(dataset, Segment.TRAIN, n_quantiles=n_quantiles)

print('\n提取验证数据...')
X_valid, y_valid, meta_valid, group_valid = extract_lambdarank_data(dataset, Segment.VALID, n_quantiles=n_quantiles)

print('\n提取测试数据...')
X_test, y_test, meta_test, group_test = extract_lambdarank_data(dataset, Segment.TEST, n_quantiles=n_quantiles)



提取 LambdaRank 训练数据...
TRAIN: 标签唯一值 = [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29]
TRAIN: X.shape=(437100, 65), y 取值 [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29]
交易日数量 = 1457, 平均每天样本数 = 300.0

提取验证数据...
VALID: 标签唯一值 = [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29]
VALID: X.shape=(72600, 65), y 取值 [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29]
交易日数量 = 242, 平均每天样本数 = 300.0

提取测试数据...
TEST: 标签唯一值 = [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29]
TEST: X.shape=(95100, 65), y 取值 [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29]
交易日数量 = 317, 平均每天样本数 = 300.0


In [9]:
# ============================================================================
# Cell 5: 训练 LambdaRank 模型
# ============================================================================
print('\n开始训练 LambdaRank 模型...')

# 创建 Dataset，直接传入 group 参数
train_data = lgb.Dataset(X_train, label=y_train, group=group_train)
valid_data = lgb.Dataset(X_valid, label=y_valid, group=group_valid, reference=train_data)

# LambdaRank 专用参数（重点：objective, metric, label_gain）
params = {
    'objective': 'lambdarank',
    'metric': 'ndcg',                 # 评估指标
    'ndcg_eval_at': [1, 3, 5, 7, 10, 20, 30],        # 计算 NDCG@1, @3, @5
    'label_gain': [i**2-1 for i in range(n_quantiles)],   # 对应 5 档标签的增益 (2^rel - 1)
    'lambdarank_truncation_level': 10, # 截断级别

    #
    'num_leaves': 1024,
    'max_depth': -1,
    'min_data_in_leaf': 300,

    # 学习参数
    'learning_rate': 0.001,
    'feature_fraction': 0.88,
    'bagging_fraction': 0.87,
    'bagging_freq': 5,

    # 正则化
    'lambda_l1': 30,
    'lambda_l2': 0.0,

    # 其他
    'boosting_type': 'gbdt',
    'device': 'gpu',
    'verbose': -1,
    'seed': 42,
    'num_threads': -1
}

num_boost_round = 1000
early_stopping_rounds = 100

# 训练（这里仍然可以用 feval 监控 IC，但不是必须）
model = lgb.train(
    params,
    train_data,
    num_boost_round=num_boost_round,
    valid_sets=[train_data, valid_data],
    valid_names=['train', 'valid'],
    callbacks=[
        lgb.early_stopping(early_stopping_rounds),
        lgb.log_evaluation(period=1)
    ]
)

print(f'\n训练完成！最佳迭代轮数: {model.best_iteration}')
if model.best_score:
    print(f"最佳验证 NDCG: {model.best_score['valid']}")


开始训练 LambdaRank 模型...
[1]	train's ndcg@1: 0.35305	train's ndcg@3: 0.352921	train's ndcg@5: 0.358231	train's ndcg@7: 0.361153	train's ndcg@10: 0.362356	train's ndcg@20: 0.374712	train's ndcg@30: 0.383265	valid's ndcg@1: 0.291593	valid's ndcg@3: 0.3336	valid's ndcg@5: 0.344224	valid's ndcg@7: 0.343533	valid's ndcg@10: 0.344112	valid's ndcg@20: 0.3467	valid's ndcg@30: 0.349131
Training until validation scores don't improve for 100 rounds
[2]	train's ndcg@1: 0.422688	train's ndcg@3: 0.419412	train's ndcg@5: 0.411705	train's ndcg@7: 0.408058	train's ndcg@10: 0.402112	train's ndcg@20: 0.400589	train's ndcg@30: 0.405114	valid's ndcg@1: 0.308294	valid's ndcg@3: 0.327482	valid's ndcg@5: 0.338007	valid's ndcg@7: 0.338643	valid's ndcg@10: 0.339793	valid's ndcg@20: 0.349685	valid's ndcg@30: 0.351327
[3]	train's ndcg@1: 0.441912	train's ndcg@3: 0.435493	train's ndcg@5: 0.427144	train's ndcg@7: 0.422001	train's ndcg@10: 0.411506	train's ndcg@20: 0.406903	train's ndcg@30: 0.409427	valid's ndcg@1: 0.

In [10]:
# ============================================================================
# Cell 6: 特征重要性分析
# ============================================================================

print('\n特征重要性分析...')
lag_days = 0
# 获取特征重要性
importance = model.feature_importance(importance_type='gain')
feature_names = [f'{factor}_lag_{lag}' for factor in FACTOR_NAMES for lag in range(0, lag_days + 1)]

# 创建重要性 DataFrame
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importance
}).sort_values('importance', ascending=False)

print('Top 20 重要特征:')
print(importance_df.head(120))


特征重要性分析...
Top 20 重要特征:
                         feature   importance
29                    rstr_lag_0  2826.416638
43              liq_amihud_lag_0  1545.668609
45   range_adjusted_amihud_lag_0   806.939310
39            turnover_vol_lag_0   780.796011
37                    cmra_lag_0   764.010643
..                           ...          ...
32  overnight_intraday_rev_lag_0     2.849100
11   corr_volume_amplitude_lag_0     1.625475
7             volume_perc5_lag_0     1.566383
4             volume_perc2_lag_0     1.501925
10   early_corr_volume_ret_lag_0     0.079475

[65 rows x 2 columns]


In [11]:
# ============================================================================
# Cell 7: 生成回测信号
# ============================================================================
print('\n在测试集上预测...')

# 预测
predictions = model.predict(X_test, num_iteration=model.best_iteration)

print(f'预测完成，预测样本数: {len(predictions)}')

# 构建信号 DataFrame
signal = meta_test.with_columns([
    pl.Series('signal', predictions)
])

print(f'\n信号数据形状: {signal.shape}')
print('信号数据预览:')
print(signal.tail(10))


在测试集上预测...
预测完成，预测样本数: 95100

信号数据形状: (95100, 3)
信号数据预览:
shape: (10, 3)
┌─────────────────────┬────────────┬───────────┐
│ datetime            ┆ vt_symbol  ┆ signal    │
│ ---                 ┆ ---        ┆ ---       │
│ datetime[μs]        ┆ str        ┆ f64       │
╞═════════════════════╪════════════╪═══════════╡
│ 2026-04-27 00:00:00 ┆ 688169.SSE ┆ -0.003358 │
│ 2026-04-27 00:00:00 ┆ 688187.SSE ┆ -0.00348  │
│ 2026-04-27 00:00:00 ┆ 688223.SSE ┆ -0.003733 │
│ 2026-04-27 00:00:00 ┆ 688256.SSE ┆ 0.002338  │
│ 2026-04-27 00:00:00 ┆ 688271.SSE ┆ -0.00303  │
│ 2026-04-27 00:00:00 ┆ 688303.SSE ┆ -0.00348  │
│ 2026-04-27 00:00:00 ┆ 688396.SSE ┆ -0.003307 │
│ 2026-04-27 00:00:00 ┆ 688472.SSE ┆ -0.003111 │
│ 2026-04-27 00:00:00 ┆ 688506.SSE ┆ -0.003358 │
│ 2026-04-27 00:00:00 ┆ 688981.SSE ┆ -0.00121  │
└─────────────────────┴────────────┴───────────┘


In [12]:
# ============================================================================
# Cell 8: 保存模型和信号
# ============================================================================
MODEL_NAME = 'v100'
SIGNAL_NAME = 'v100'

lab.save_model(MODEL_NAME, model)
lab.save_signal(SIGNAL_NAME, signal)
# # 保存 LightGBM 模型
# MODEL_PICKLE_PATH = LAB_PATH / 'model' / f'{MODEL_NAME}.pkl'
# with open(MODEL_PICKLE_PATH, 'wb') as f:
#     pickle.dump({
#         'model': model,
#         'params': params,
#         'best_iteration': model.best_iteration,
#         'best_score': model.best_score
#     }, f)
# print(f'模型已保存: {MODEL_PICKLE_PATH}')
#
# # 保存信号
# SIGNAL_PARQUET_PATH = LAB_PATH / 'signal' / f'{SIGNAL_NAME}.parquet'
# SIGNAL_PARQUET_PATH.parent.mkdir(parents=True, exist_ok=True)
# signal.write_parquet(str(SIGNAL_PARQUET_PATH))
# print(f'信号已保存: {SIGNAL_PARQUET_PATH}')

In [13]:
with pd.option_context('display.max_rows', None):
    print(importance_df.head(120))

                            feature   importance
29                       rstr_lag_0  2826.416638
43                 liq_amihud_lag_0  1545.668609
45      range_adjusted_amihud_lag_0   806.939310
39               turnover_vol_lag_0   780.796011
37                       cmra_lag_0   764.010643
44          zero_trades_ratio_lag_0   430.532309
55                  gap_ratio_lag_0   347.672479
62                daily_range_lag_0   337.932812
12                 mmt_last30_lag_0   327.284997
38               abn_turnover_lag_0   312.832653
52       high_position_volume_lag_0   170.037139
2            corr_ret_lastret_lag_0   166.673829
60              ma_divergence_lag_0   137.890276
63           effective_spread_lag_0   127.936579
53        low_position_volume_lag_0   114.560303
36              vol_downRatio_lag_0   104.062626
46                     vr_20d_lag_0    86.143160
1               down_vol_perc_lag_0    83.873992
26            ideal_range_cut_lag_0    79.972631
51                  